# Plot position-coded nodal RSAM time series

This notebook replaces both:

- `60_plot_nodal_rsam.py`
- `60_run_plot_nodal_rsam.sh`

It reads the recomputed RSAM archive at:

```text
/Volumes/tachyon/LBSSP_DATA/nodal_rsam_position_codes
```

and produces conventional FLOVOpy `RSAM.plot()` figures for T1 and T3.

Because step 86 standardized the nodal data, only these channels are used:

```text
DPE
DPN
DPZ
```

The station code is already the node position in centimetres, so no legacy
serial-number mapping is needed.

The new RSAM computation used these named frequency bands:

```text
B4_8, B8_16, B16_32, B32_64, B64_128, B128_240
```

The old wrapper's `LOW_5_20`, `MID_20_80`, and `HIGH_80_240` metrics are
therefore not requested here.

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Optional

import pandas as pd
from obspy import UTCDateTime

from flovopy.processing.sam import RSAM

## Configuration

The default workflow creates one plot set for every network, deployment
location, and DP channel that actually exists. Empty combinations are skipped.

In [2]:
SAM_ROOT = Path(
    "/Volumes/tachyon/LBSSP_DATA/nodal_rsam_position_codes"
)
PLOT_DIR = SAM_ROOT / "plots"

START = UTCDateTime("2026-05-16T00:00:00")
END = UTCDateTime("2026-05-20T00:00:00")

NETWORK_LOCATIONS = {
    "T1": ["N1", "N2", "N3"],
    "T3": ["N4"],
}

CHANNELS = ["DPE", "DPN", "DPZ"]

SAMPLING_INTERVAL_S = 60
RSAM_EXTENSION = "csv"

# Metrics created by the recomputation notebook.
METRICS = [
    "mean",
    "median",
    "rms",
    "B4_8",
    "B8_16",
    "B16_32",
    "B32_64",
    "B64_128",
    "B128_240",
]

PLOT_KIND = "line"       # "stream", "line", or "scatter"
LOG_Y = False
EQUAL_SCALE = True
Y_LIMITS = None          # for example: (0.0, 1000.0)
TRIM_TO_DATA = True

VERBOSE_READ = False

PLOT_DIR.mkdir(parents=True, exist_ok=True)

print(f"RSAM root: {SAM_ROOT}")
print(f"Plot dir:  {PLOT_DIR}")
print(f"Time:      {START} to {END}")

RSAM root: /Volumes/tachyon/LBSSP_DATA/nodal_rsam_position_codes
Plot dir:  /Volumes/tachyon/LBSSP_DATA/nodal_rsam_position_codes/plots
Time:      2026-05-16T00:00:00.000000Z to 2026-05-20T00:00:00.000000Z


## Helper functions

In [3]:
def parse_seed_id(seed_id: str) -> tuple[str, str, str, str]:
    """Split NET.STA.LOC.CHA and validate its structure."""
    parts = seed_id.split(".")
    if len(parts) != 4:
        raise ValueError(
            f"Expected NET.STA.LOC.CHA, got {seed_id!r}"
        )
    return tuple(parts)


def station_code_to_position_m(station: str) -> float:
    """Convert a position-coded station name in centimetres to metres."""
    text = str(station).strip()

    if text.endswith(".0"):
        text = text[:-2]

    if not text.isdigit():
        raise ValueError(
            f"Station code {station!r} is not an integer centimetre position."
        )

    return int(text) / 100.0


def summarize_rsam(
    rsam: RSAM,
    *,
    max_ids: int = 12,
) -> pd.DataFrame:
    """Return and print a compact inventory of an RSAM object."""
    rows = []

    for seed_id, dataframe in getattr(
        rsam,
        "dataframes",
        {},
    ).items():
        try:
            network, station, location, channel = (
                parse_seed_id(seed_id)
            )
            position_m = station_code_to_position_m(station)
        except Exception:
            network = station = location = channel = ""
            position_m = float("nan")

        rows.append(
            {
                "seed_id": seed_id,
                "network": network,
                "station": station,
                "position_m": position_m,
                "location": location,
                "channel": channel,
                "rows": (
                    len(dataframe)
                    if dataframe is not None
                    else 0
                ),
                "columns": (
                    ", ".join(map(str, dataframe.columns))
                    if dataframe is not None
                    else ""
                ),
            }
        )

    summary = pd.DataFrame(rows)

    print("=" * 80)
    print(f"Number of trace IDs: {len(summary)}")

    if summary.empty:
        print("No dataframes loaded.")
    else:
        print(summary.head(max_ids).to_string(index=False))

        if len(summary) > max_ids:
            print(f"... {len(summary) - max_ids} more trace IDs")

        print(
            "Distinct positions: "
            f"{summary['position_m'].nunique(dropna=True)}"
        )
        print(
            "Locations: "
            f"{sorted(summary['location'].dropna().unique())}"
        )
        print(
            "Channels: "
            f"{sorted(summary['channel'].dropna().unique())}"
        )

    print("=" * 80)
    return summary


def select_exact_ids(
    rsam: RSAM,
    *,
    network: Optional[str] = None,
    station: Optional[str] = None,
    location: Optional[str] = None,
    channel: Optional[str] = None,
) -> RSAM:
    """Select RSAM IDs by exact SEED fields."""
    selected_ids = []

    for seed_id in getattr(rsam, "dataframes", {}):
        try:
            net, sta, loc, cha = parse_seed_id(seed_id)
        except ValueError:
            continue

        if network is not None and net != network:
            continue
        if station is not None and sta != station:
            continue
        if location is not None and loc != location:
            continue
        if channel is not None and cha != channel:
            continue

        selected_ids.append(seed_id)

    return rsam.select_ids(selected_ids)


def available_metrics(rsam: RSAM) -> list[str]:
    """Return metrics shared by all non-empty dataframes."""
    column_sets = []

    for dataframe in getattr(rsam, "dataframes", {}).values():
        if dataframe is not None and len(dataframe):
            column_sets.append(set(map(str, dataframe.columns)))

    if not column_sets:
        return []

    shared = set.intersection(*column_sets)
    return sorted(shared)


def choose_metrics(
    rsam: RSAM,
    requested_metrics: list[str],
) -> tuple[list[str], list[str]]:
    """Split requested metrics into available and unavailable names."""
    available = set(available_metrics(rsam))
    selected = [
        metric
        for metric in requested_metrics
        if metric in available
    ]
    missing = [
        metric
        for metric in requested_metrics
        if metric not in available
    ]
    return selected, missing

## Load each network once

The RSAM archive is read once per network, then subset in memory for each
location and channel.

In [4]:
rsam_by_network: dict[str, RSAM] = {}
network_inventories: dict[str, pd.DataFrame] = {}

for network in NETWORK_LOCATIONS:
    print(
        f"\nReading {network}: {START} to {END}"
    )

    rsam = RSAM.read(
        START,
        END,
        SAM_DIR=str(SAM_ROOT),
        network=network,
        sampling_interval=SAMPLING_INTERVAL_S,
        ext=RSAM_EXTENSION,
        verbose=VERBOSE_READ,
    )

    rsam_by_network[network] = rsam
    network_inventories[network] = summarize_rsam(rsam)


Reading T1: 2026-05-16T00:00:00.000000Z to 2026-05-20T00:00:00.000000Z
Dataframe with 1376 rows is already on a regular 60 s grid based on column 'time'
Dataframe with 1376 rows is already on a regular 60 s grid based on column 'time'
Dataframe with 1376 rows is already on a regular 60 s grid based on column 'time'
Dataframe with 70 rows is already on a regular 60 s grid based on column 'time'
Dataframe with 70 rows is already on a regular 60 s grid based on column 'time'
Dataframe with 70 rows is already on a regular 60 s grid based on column 'time'
Dataframe with 1376 rows is already on a regular 60 s grid based on column 'time'
Dataframe with 1376 rows is already on a regular 60 s grid based on column 'time'
Dataframe with 1376 rows is already on a regular 60 s grid based on column 'time'
Dataframe with 69 rows is already on a regular 60 s grid based on column 'time'
Dataframe with 69 rows is already on a regular 60 s grid based on column 'time'
Dataframe with 69 rows is already on

## Check metric availability

This prevents the plotting loop from failing if a requested metric is absent
from a particular subset.

In [6]:
metric_inventory_rows = []

for network, rsam in rsam_by_network.items():
    metrics = available_metrics(rsam)
    print(metrics)

    metric_inventory_rows.append(
        {
            "network": network,
            "available_metrics": ", ".join(metrics),
        }
    )

metric_inventory = pd.DataFrame(metric_inventory_rows)
metric_inventory

['B128_240', 'B16_32', 'B32_64', 'B4_8', 'B64_128', 'B8_16', 'max', 'mean', 'median', 'min', 'rms', 'time']
['B128_240', 'B16_32', 'B32_64', 'B4_8', 'B64_128', 'B8_16', 'max', 'mean', 'median', 'min', 'rms', 'time']


,network,available_metrics
0,T1,"B128_240, B16_32, B32_64, B4_8, B64_128, B8_16..."
1,T3,"B128_240, B16_32, B32_64, B4_8, B64_128, B8_16..."


## Plot every available network/location/channel subset

For `kind="line"` or `kind="scatter"`, FLOVOpy may write one figure containing
the requested metrics. For `kind="stream"`, it may write one file per metric,
depending on the installed FLOVOpy version.

The notebook records each attempted output base and skips selectors containing
no RSAM data.

In [7]:
plot_summary_rows = []

for network, locations in NETWORK_LOCATIONS.items():
    network_rsam = rsam_by_network[network]

    for location in locations:
        for channel in CHANNELS:
            print("\n" + "=" * 88)
            print(
                f"Plotting network={network}, "
                f"location={location}, channel={channel}"
            )
            print("=" * 88)

            selected = select_exact_ids(
                network_rsam,
                network=network,
                location=location,
                channel=channel,
            )

            n_ids = len(
                getattr(selected, "dataframes", {})
            )

            if n_ids == 0:
                print("No matching RSAM IDs; skipping.")
                plot_summary_rows.append(
                    {
                        "network": network,
                        "location": location,
                        "channel": channel,
                        "trace_ids": 0,
                        "positions": 0,
                        "metrics": "",
                        "missing_metrics": "",
                        "output_base": "",
                        "status": "no data",
                    }
                )
                continue

            subset_summary = summarize_rsam(selected)

            selected_metrics, missing_metrics = choose_metrics(
                selected,
                METRICS,
            )

            if missing_metrics:
                print(
                    "Unavailable requested metrics: "
                    + ", ".join(missing_metrics)
                )

            if not selected_metrics:
                print("No requested metrics are available; skipping.")
                plot_summary_rows.append(
                    {
                        "network": network,
                        "location": location,
                        "channel": channel,
                        "trace_ids": n_ids,
                        "positions": (
                            subset_summary["position_m"]
                            .nunique(dropna=True)
                        ),
                        "metrics": "",
                        "missing_metrics": ", ".join(
                            missing_metrics
                        ),
                        "output_base": "",
                        "status": "no requested metrics",
                    }
                )
                continue

            outfile = (
                PLOT_DIR
                / f"{network}_{location}_{channel}_rsam.png"
            )

            print(
                "Plotting metrics: "
                + ", ".join(selected_metrics)
            )
            print(f"Output base: {outfile}")

            try:
                selected.plot(
                    metrics=selected_metrics,
                    kind=PLOT_KIND,
                    logy=LOG_Y,
                    equal_scale=EQUAL_SCALE,
                    outfile=str(outfile),
                    ylims=Y_LIMITS,
                    trim_to_data=TRIM_TO_DATA,
                )
                status = "written"
                print("Plot completed.")

            except Exception as exc:
                status = f"failed: {exc}"
                print(status)

            plot_summary_rows.append(
                {
                    "network": network,
                    "location": location,
                    "channel": channel,
                    "trace_ids": n_ids,
                    "positions": (
                        subset_summary["position_m"]
                        .nunique(dropna=True)
                    ),
                    "metrics": ", ".join(selected_metrics),
                    "missing_metrics": ", ".join(
                        missing_metrics
                    ),
                    "output_base": str(outfile),
                    "status": status,
                }
            )

plot_summary = pd.DataFrame(plot_summary_rows)
plot_summary


Plotting network=T1, location=N1, channel=DPE
Number of trace IDs: 35
        seed_id network station  position_m location channel  rows                                                                           columns
T1.02800.N1.DPE      T1   02800       28.00       N1     DPE  1376 time, min, mean, max, median, rms, B4_8, B8_16, B16_32, B32_64, B64_128, B128_240
T1.03600.N1.DPE      T1   03600       36.00       N1     DPE  1376 time, min, mean, max, median, rms, B4_8, B8_16, B16_32, B32_64, B64_128, B128_240
T1.04400.N1.DPE      T1   04400       44.00       N1     DPE  1377 time, min, mean, max, median, rms, B4_8, B8_16, B16_32, B32_64, B64_128, B128_240
T1.05207.N1.DPE      T1   05207       52.07       N1     DPE  1364 time, min, mean, max, median, rms, B4_8, B8_16, B16_32, B32_64, B64_128, B128_240
T1.06004.N1.DPE      T1   06004       60.04       N1     DPE  1357 time, min, mean, max, median, rms, B4_8, B8_16, B16_32, B32_64, B64_128, B128_240
T1.06809.N1.DPE      T1   06809    

,network,location,channel,trace_ids,positions,metrics,missing_metrics,output_base,status
0,T1,N1,DPE,35,35,"mean, median, rms, B4_8, B8_16, B16_32, B32_64...",,/Volumes/tachyon/LBSSP_DATA/nodal_rsam_positio...,written
1,T1,N1,DPN,35,35,"mean, median, rms, B4_8, B8_16, B16_32, B32_64...",,/Volumes/tachyon/LBSSP_DATA/nodal_rsam_positio...,written
2,T1,N1,DPZ,35,35,"mean, median, rms, B4_8, B8_16, B16_32, B32_64...",,/Volumes/tachyon/LBSSP_DATA/nodal_rsam_positio...,written
3,T1,N2,DPE,35,35,"mean, median, rms, B4_8, B8_16, B16_32, B32_64...",,/Volumes/tachyon/LBSSP_DATA/nodal_rsam_positio...,written
4,T1,N2,DPN,35,35,"mean, median, rms, B4_8, B8_16, B16_32, B32_64...",,/Volumes/tachyon/LBSSP_DATA/nodal_rsam_positio...,written
5,T1,N2,DPZ,35,35,"mean, median, rms, B4_8, B8_16, B16_32, B32_64...",,/Volumes/tachyon/LBSSP_DATA/nodal_rsam_positio...,written
6,T1,N3,DPE,35,35,"mean, median, rms, B4_8, B8_16, B16_32, B32_64...",,/Volumes/tachyon/LBSSP_DATA/nodal_rsam_positio...,written
7,T1,N3,DPN,35,35,"mean, median, rms, B4_8, B8_16, B16_32, B32_64...",,/Volumes/tachyon/LBSSP_DATA/nodal_rsam_positio...,written
8,T1,N3,DPZ,35,35,"mean, median, rms, B4_8, B8_16, B16_32, B32_64...",,/Volumes/tachyon/LBSSP_DATA/nodal_rsam_positio...,written
9,T3,N4,DPE,12,12,"mean, median, rms, B4_8, B8_16, B16_32, B32_64...",,/Volumes/tachyon/LBSSP_DATA/nodal_rsam_positio...,written


## Compact completion summary

In [ ]:
completion_summary = (
    plot_summary
    .groupby(
        ["network", "location", "channel", "status"],
        dropna=False,
    )
    .agg(
        trace_ids=("trace_ids", "max"),
        positions=("positions", "max"),
        metrics=("metrics", "first"),
        output_base=("output_base", "first"),
    )
    .reset_index()
    .sort_values(
        ["network", "location", "channel"]
    )
)

completion_summary

## Optional: plot all deployments of one network/channel together

Because station codes encode physical position, this optional cell combines
N1/N2/N3 for T1 or N4 for T3 into one conventional RSAM line plot. It does not
create the position–time heatmaps from notebook 65; it simply lets
`RSAM.plot()` draw every matching position-coded trace together.

In [ ]:
COMBINED_NETWORK = "T1"
COMBINED_CHANNEL = "DPZ"
RUN_COMBINED_EXAMPLE = False

if RUN_COMBINED_EXAMPLE:
    combined = select_exact_ids(
        rsam_by_network[COMBINED_NETWORK],
        network=COMBINED_NETWORK,
        channel=COMBINED_CHANNEL,
    )

    selected_metrics, missing_metrics = choose_metrics(
        combined,
        METRICS,
    )

    if not selected_metrics:
        raise RuntimeError(
            "No requested metrics are available for the combined plot."
        )

    combined_outfile = (
        PLOT_DIR
        / (
            f"{COMBINED_NETWORK}_all_locations_"
            f"{COMBINED_CHANNEL}_rsam.png"
        )
    )

    combined.plot(
        metrics=selected_metrics,
        kind=PLOT_KIND,
        logy=LOG_Y,
        equal_scale=EQUAL_SCALE,
        outfile=str(combined_outfile),
        ylims=Y_LIMITS,
        trim_to_data=TRIM_TO_DATA,
    )

    print(f"Wrote combined plot base: {combined_outfile}")
else:
    print(
        "Set RUN_COMBINED_EXAMPLE = True to create "
        "the optional combined plot."
    )